Made this notebook to check that I get the same results as Gallagher et al on their parameter estiamtes directly.
After that, we investigate if median stan draws line up with their estimates. 

In [2]:
import pandas as pd
import sys

sys.path.append("..")  

from Mathematical_Biomarkers.utils.myUtils import calc_average_overshoot, calc_delta_benefit, predict_ct_ttp, predict_drug_benefit


Have to redefine predict eTTP function since its not in myutils

In [3]:
def predict_at_window_ttp(tau, n_crit, **kwargs):
    params = kwargs
    n_overshoot = calc_average_overshoot(tau, n_crit, **params)
    ave_n = ((n_crit + 1.2 * params['n0'])/2 - n_overshoot)
    benefit = calc_delta_benefit(kappa=(ave_n / params['K']), **params)
    ct_ttp = predict_ct_ttp(**params)
    return ct_ttp * (1 + benefit)

In [4]:
first_cycle_bruchovsky = pd.read_csv("../Mathematical_Biomarkers/paper_figures/patient_fits/first_cycle_params.csv")
first_cycle_bruchovsky['R0']

0    0.00001
1    0.00001
2    0.00001
3    0.00001
4    0.00001
Name: R0, dtype: float64

Biomarker values from the From table 8 in biomarker supplemental file

| Patient ID | AT Score | eMDD  | eTTP, d |
|:----------:|:--------:|:-----:|:-------:|
| 12         | 0.270    | 0.625 | 588     |
| 20         | 0.495    | 0.352 | 1144    |
| 25         | 0.439    | 0.362 | 1292    |
| 78         | 0.552    | 0.311 | 1235    |
| 85         | 0.366    | 0.449 | 907     |

We recalculate let them using what i assume to be the same functions as the original manuscript


In [ ]:
ids = [12,20,25,78,85]
filtered_ids = first_cycle_bruchovsky[first_cycle_bruchovsky['PatientId'].isin(ids)] 

filtered_ids["at_score"] = calc_delta_benefit(kappa=.2, **filtered_ids)
filtered_ids['eTTP'] = predict_at_window_ttp(tau=21, n_crit=0.5 * filtered_ids.n0, **filtered_ids)
filtered_ids['eMDD'] = predict_drug_benefit(tau=21, n_crit=0.5 * filtered_ids.n0, **filtered_ids)

filtered_ids[['PatientId', 'at_score', 'eMDD', 'eTTP']]


,PatientId,at_score,eMDD,eTTP
0,12,0.270091,0.624651,630.111572
1,20,0.495319,0.323769,1825.542787
2,25,0.438991,0.368763,1532.583156
3,78,0.551727,0.287482,2159.476669
4,85,0.366395,0.446532,1199.892574
